In [1]:
# ============================================================
# NACC ALZHEIMER — 5 MODEL RELEASE PIPELINE (PARTS 0–8)
# - Uses frozen clean dataset: nacc_clean_complete_case_AD01.csv
# - 50 seeds, stratified 80/20
# - Models: XGBoost, LogReg, Ridge, KNN, Linear SVM (SVM last)
# - Includes: accuracy barplot, top10 importance barplots,
#            canonical metrics + tests, calibration, redundancy + plots
# - Excludes: heatmaps
# - Saves EVERYTHING to: ~/Research/Alzheimer/5 model/run_YYYYMMDD_HHMMSS/
# ============================================================

import json
from pathlib import Path
import numpy as np
import pandas as pd
from tqdm.auto import tqdm
import matplotlib.pyplot as plt
import seaborn as sns
import scipy.stats as stats

from xgboost import XGBClassifier
from sklearn import svm
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression, RidgeClassifier

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, confusion_matrix, precision_score, f1_score,
    roc_auc_score, average_precision_score, brier_score_loss
)
from sklearn.inspection import permutation_importance
from sklearn.calibration import calibration_curve
from statsmodels.stats.contingency_tables import mcnemar

sns.set(style="whitegrid")

# -----------------------------
# Pretty helpers
# -----------------------------
def print_header(title):
    print("\n" + "="*100)
    print(title)
    print("="*100)

def safe_div(a, b):
    return a / b if b != 0 else np.nan

def compute_classification_metrics(y_true, y_pred):
    cm = confusion_matrix(y_true, y_pred)
    if cm.size != 4:
        return {"Sensitivity": np.nan, "Specificity": np.nan, "Precision": np.nan, "F1": np.nan}
    tn, fp, fn, tp = cm.ravel()
    sens = safe_div(tp, tp + fn)
    spec = safe_div(tn, tn + fp)
    prec = precision_score(y_true, y_pred, zero_division=0)
    f1 = f1_score(y_true, y_pred, zero_division=0)
    return {"Sensitivity": float(sens), "Specificity": float(spec), "Precision": float(prec), "F1": float(f1)}

# -----------------------------
# DeLong (correlated ROC-AUC test)
# -----------------------------
def _compute_midrank(x):
    J = np.argsort(x)
    Z = x[J]
    N = len(x)
    T = np.zeros(N, dtype=float)
    i = 0
    while i < N:
        j = i
        while j < N and Z[j] == Z[i]:
            j += 1
        T[i:j] = 0.5 * (i + j - 1)
        i = j
    T2 = np.empty(N, dtype=float)
    T2[J] = T + 1
    return T2

def _fastDeLong(predictions_sorted_transposed, label_1_count):
    m = label_1_count
    n = predictions_sorted_transposed.shape[1] - m

    pos = predictions_sorted_transposed[:, :m]
    neg = predictions_sorted_transposed[:, m:]
    k = predictions_sorted_transposed.shape[0]

    tx = np.empty([k, m], dtype=float)
    ty = np.empty([k, n], dtype=float)
    tz = np.empty([k, m + n], dtype=float)

    for r in range(k):
        tx[r, :] = _compute_midrank(pos[r, :])
        ty[r, :] = _compute_midrank(neg[r, :])
        tz[r, :] = _compute_midrank(predictions_sorted_transposed[r, :])

    aucs = tz[:, :m].sum(axis=1) / (m * n) - (m + 1.0) / (2.0 * n)
    v01 = (tx / n) - (tz[:, :m] / n)
    v10 = 1.0 - (ty / m) + (tz[:, m:] / m)

    sx = np.cov(v01)
    sy = np.cov(v10)
    s = sx / m + sy / n
    return aucs, s

def delong_roc_test(y_true, y_scores_a, y_scores_b):
    y_true = np.asarray(y_true).astype(int)
    order = np.argsort(-y_true)
    y_sorted = y_true[order]
    preds_sorted = np.vstack([np.asarray(y_scores_a)[order], np.asarray(y_scores_b)[order]])

    m = int(np.sum(y_sorted))
    aucs, cov = _fastDeLong(preds_sorted, m)
    diff = aucs[0] - aucs[1]
    var = cov[0, 0] + cov[1, 1] - 2 * cov[0, 1]
    if var <= 0:
        return np.nan, np.nan, float(aucs[0]), float(aucs[1])
    z = diff / np.sqrt(var)
    p = 2 * (1 - stats.norm.cdf(abs(z)))
    return float(z), float(p), float(aucs[0]), float(aucs[1])

# -----------------------------
# McNemar
# -----------------------------
def mcnemar_test(y_true, y_pred_a, y_pred_b):
    y_true = np.asarray(y_true)
    y_pred_a = np.asarray(y_pred_a)
    y_pred_b = np.asarray(y_pred_b)

    both_correct = np.sum((y_pred_a == y_true) & (y_pred_b == y_true))
    a_correct_b_wrong = np.sum((y_pred_a == y_true) & (y_pred_b != y_true))
    a_wrong_b_correct = np.sum((y_pred_a != y_true) & (y_pred_b == y_true))
    both_wrong = np.sum((y_pred_a != y_true) & (y_pred_b != y_true))

    table = [[both_correct, a_correct_b_wrong],
             [a_wrong_b_correct, both_wrong]]
    res = mcnemar(table, exact=False, correction=True)
    return float(res.statistic), float(res.pvalue), table

# ============================================================
# PART 0) LOAD CLEAN FROZEN DATA + OUTPUT FOLDER (AD03 Gaussian)
# ============================================================
print_header("PART 0) LOAD CLEAN FROZEN DATA (AD03 Gaussian) + CREATE OUTPUT FOLDER")

BASE_DIR = Path.home() / "Research" / "Alzheimer"

# Gaussian-noise dataset
DATA_PATH = BASE_DIR / "nacc_clean_complete_case_AD03.csv"

TARGET = "NACCALZD"

if not DATA_PATH.exists():
    raise FileNotFoundError(f"Could not find dataset at: {DATA_PATH}")

df = pd.read_csv(DATA_PATH, low_memory=False)

# Keep definitive labels only
df = df[df[TARGET].isin([0, 1])].copy()
df[TARGET] = df[TARGET].astype(int)


# Drop leakage / ID / visit metadata
DROP_COLS = [
    TARGET,

    # label-derived flags
    "ad_flag", "mci_flag", "dementia_flag",
    "DEMENTED", "NACCTMCI", "NORMCOG",

    # identifiers / visit metadata
    "NACCID", "NACCVNUM",
    "VISITYR", "VISITMO", "VISITDAY",
    "BIRTHYR", "BIRTHMO",

    # removed during AD02 construction
    "AMYLPET", "TAUPETAD", "FDGAD",
    "AMYLCSF", "CSFTAU",

    "NACCMMS", "NACCMOCA",
    "CDRSUM", "CDRGLOB",
]


# Build modeling matrix
X = df.drop(columns=[c for c in DROP_COLS if c in df.columns], errors="ignore")

# Numeric predictors only
X = X.select_dtypes(include=[np.number]).copy()

y = df[TARGET].copy()

feature_names = X.columns.tolist()


# Output folder
timestamp = pd.Timestamp.now().strftime("%Y%m%d_%H%M%S")

OUT_DIR = BASE_DIR / "5 model" / f"run_AD03_gaussian_{timestamp}"

(OUT_DIR / "figures").mkdir(parents=True, exist_ok=True)
(OUT_DIR / "tables").mkdir(parents=True, exist_ok=True)


print("✅ Loaded:", DATA_PATH)
print("Shape df:", df.shape)
print("Shape X :", X.shape)
print("Positive rate:", float(np.mean(y == 1)))
print("Features:", len(feature_names))
print("✅ Output folder:", OUT_DIR)


# Save manifest early
manifest0 = {
    "data_path": str(DATA_PATH),
    "out_dir": str(OUT_DIR),
    "dataset": "AD03_gaussian_noise",

    "n_rows": int(df.shape[0]),
    "n_features": int(X.shape[1]),
    "target": TARGET,

    "dropped_cols_present": [c for c in DROP_COLS if c in df.columns],
}

with open(OUT_DIR / "manifest_part0.json", "w") as f:
    json.dump(manifest0, f, indent=2)
# ============================================================
# MODELS (5 only; SVM last)
# ============================================================
def build_models(seed=0):
    return {
        "XGBoost": XGBClassifier(
            tree_method="hist",
            random_state=seed,
            n_estimators=300,
            max_depth=4,
            learning_rate=0.08,
            subsample=0.9,
            colsample_bytree=0.9,
            eval_metric="logloss",
            n_jobs=-1
        ),
        "Logistic Regression": Pipeline([
            ("scaler", StandardScaler()),
            ("clf", LogisticRegression(max_iter=10000, random_state=seed))
        ]),
        "Ridge Classifier": Pipeline([
            ("scaler", StandardScaler()),
            ("clf", RidgeClassifier(random_state=seed))
        ]),
        "KNN": Pipeline([
            ("scaler", StandardScaler()),
            ("clf", KNeighborsClassifier(n_neighbors=5))
        ]),
        # SLOWEST LAST
        "SVM (Linear)": Pipeline([
            ("scaler", StandardScaler()),
            ("clf", svm.SVC(kernel="linear", probability=True, random_state=seed))
        ]),
    }

MODEL_ORDER = list(build_models(0).keys())

# ============================================================
# PART 1) ACCURACY ACROSS 50 SEEDS (+ save canonical split)
# ============================================================
print_header("PART 1) ACCURACY ACROSS 50 SEEDS (80/20 stratified) — SVM LAST")

SEEDS = list(range(50))
canonical_seed = SEEDS[-1]

acc_seed = {m: [] for m in MODEL_ORDER}
X_train = X_test = y_train = y_test = None

pbar = tqdm(SEEDS, desc="Part 1: 50 seeds (accuracy)", leave=True)

for seed in pbar:
    Xt, Xv, yt, yv = train_test_split(
        X, y, test_size=0.2, stratify=y, random_state=seed
    )
    models = build_models(seed)

    # run in batches: loop in fixed order (SVM last)
    for name in MODEL_ORDER:
        model = models[name]
        model.fit(Xt, yt)
        pred = model.predict(Xv)
        acc_seed[name].append(float(accuracy_score(yv, pred)))

    if seed == canonical_seed:
        X_train, X_test, y_train, y_test = Xt, Xv, yt, yv

seed_acc_df = pd.DataFrame(acc_seed)
seed_acc_path = OUT_DIR / "tables" / "seed_accuracies_5models.csv"
seed_acc_df.to_csv(seed_acc_path, index=False)
print("✅ Saved:", seed_acc_path)

# Accuracy summary
acc_summary = pd.DataFrame({
    "Model": MODEL_ORDER,
    "mean_acc": [seed_acc_df[m].mean() for m in MODEL_ORDER],
    "sd_acc":   [seed_acc_df[m].std(ddof=1) for m in MODEL_ORDER],
}).sort_values("mean_acc", ascending=False)
acc_summary_path = OUT_DIR / "tables" / "accuracy_summary_mean_sd_5models.csv"
acc_summary.to_csv(acc_summary_path, index=False)
print("✅ Saved:", acc_summary_path)

# ============================================================
# PART 2) ACCURACY BAR PLOT (5 models, colored bars)
# ============================================================
print_header("PART 2) ACCURACY BAR PLOT (5 models)")

plt.figure(figsize=(10, 5))
ax = sns.barplot(data=acc_summary, x="Model", y="mean_acc", palette="tab10", ci=None)
# Add numeric labels above bars (mean accuracy)
for i, v in enumerate(acc_summary["mean_acc"].values):
    ax.text(i, v + 0.01, f"{v:.3f}", ha="center", va="bottom", fontsize=10)
ax.errorbar(
    x=np.arange(acc_summary.shape[0]),
    y=acc_summary["mean_acc"].values,
    yerr=acc_summary["sd_acc"].values,
    fmt="none",
    capsize=6,
    linewidth=2
)
ax.set_title("Accuracy Across 50 Seeds (Mean ± SD) — 5 Models")
ax.set_xlabel("")
ax.set_ylabel("Accuracy")
plt.xticks(rotation=15, ha="right")
plt.ylim(0, 1)
plt.tight_layout()

fig_acc = OUT_DIR / "figures" / "accuracy_bar_5models.png"
plt.savefig(fig_acc, dpi=300)
plt.close()
print("✅ Saved:", fig_acc)

# Paired t-tests (accuracy)
tt_rows = []
names = MODEL_ORDER
for i in range(len(names)):
    for j in range(i+1, len(names)):
        A, B = names[i], names[j]
        t, p = stats.ttest_rel(seed_acc_df[A], seed_acc_df[B])
        tt_rows.append({"A": A, "B": B, "t_stat": float(t), "p_value": float(p)})
tt_df = pd.DataFrame(tt_rows).sort_values("p_value")
tt_path = OUT_DIR / "tables" / "paired_ttests_accuracy_across_seeds_5models.csv"
tt_df.to_csv(tt_path, index=False)
print("✅ Saved:", tt_path)

# ============================================================
# PART 3) FEATURE IMPORTANCE ACROSS SEEDS (Top-10 + bar charts)
# ============================================================
print_header("PART 3) FEATURE IMPORTANCE ACROSS 50 SEEDS (Top-10 per model)")

feat_imp = {m: [] for m in MODEL_ORDER}
pbar = tqdm(SEEDS, desc="Part 3: feature importance", leave=True)

for seed in pbar:
    Xt, Xv, yt, yv = train_test_split(
        X, y, test_size=0.2, stratify=y, random_state=seed
    )
    models = build_models(seed)

    # XGBoost feature_importances_
    xgb = models["XGBoost"]
    xgb.fit(Xt, yt)
    feat_imp["XGBoost"].append(xgb.feature_importances_)

    # Logistic Regression coef_
    lr = models["Logistic Regression"]
    lr.fit(Xt, yt)
    feat_imp["Logistic Regression"].append(np.abs(lr.named_steps["clf"].coef_[0]))

    # Ridge coef_
    ridge = models["Ridge Classifier"]
    ridge.fit(Xt, yt)
    coef = ridge.named_steps["clf"].coef_
    coef = np.mean(np.abs(coef), axis=0) if coef.ndim > 1 else np.abs(coef)
    feat_imp["Ridge Classifier"].append(coef)

    # KNN permutation importance (on validation fold)
    knn = models["KNN"]
    knn.fit(Xt, yt)
    pi = permutation_importance(knn, Xv, yv, n_repeats=5, random_state=seed, n_jobs=-1)
    feat_imp["KNN"].append(np.clip(pi.importances_mean, 0, None))

    # SVM linear — use coef_ from underlying linear SVC in SVC? (SVC has coef_ for linear kernel)
    svm_m = models["SVM (Linear)"]
    svm_m.fit(Xt, yt)
    # coef_ exists for linear kernel SVC
    feat_imp["SVM (Linear)"].append(np.abs(svm_m.named_steps["clf"].coef_.ravel()))

importance_dfs = {}
for name, arrs in feat_imp.items():
    avg = np.mean(np.vstack(arrs), axis=0)
    imp_df = pd.DataFrame({"Feature": feature_names, "Importance": avg}).sort_values("Importance", ascending=False)
    importance_dfs[name] = imp_df

# Save full importance tables + top10 tables
imp_all_path = OUT_DIR / "tables" / "feature_importance_all_5models.csv"
pd.concat(
    [df.assign(Model=model) for model, df in importance_dfs.items()],
    ignore_index=True
).to_csv(imp_all_path, index=False)
print("✅ Saved:", imp_all_path)

top10_rows = []
for model, df_imp in importance_dfs.items():
    top10 = df_imp.head(10).copy()
    top10["Model"] = model
    top10_rows.append(top10)
top10_df = pd.concat(top10_rows, ignore_index=True)
top10_path = OUT_DIR / "tables" / "top10_features_5models.csv"
top10_df.to_csv(top10_path, index=False)
print("✅ Saved:", top10_path)

# Top-10 bar chart per model
for model, df_imp in importance_dfs.items():
    top = df_imp.head(10).iloc[::-1]  # reverse for barh
    plt.figure(figsize=(9, 5))
    plt.barh(top["Feature"], top["Importance"])
    plt.title(f"Top 10 Feature Importance — {model}")
    plt.xlabel("Importance")
    plt.tight_layout()
    outp = OUT_DIR / "figures" / f"top10_importance_{model.replace(' ','_').replace('(','').replace(')','')}.png"
    plt.savefig(outp, dpi=300)
    plt.close()
    print("✅ Saved:", outp)

# Combined grid figure (5 models)
fig, axes = plt.subplots(3, 2, figsize=(16, 14))
axes = axes.ravel()
for i, model in enumerate(MODEL_ORDER):
    ax = axes[i]
    top = importance_dfs[model].head(10).iloc[::-1]
    ax.barh(top["Feature"], top["Importance"])
    ax.set_title(model)
    ax.tick_params(axis="y", labelsize=8)
for j in range(len(MODEL_ORDER), len(axes)):
    axes[j].axis("off")
plt.suptitle("Top 10 Feature Importances — 5 Models", fontsize=16, weight="bold")
plt.tight_layout(rect=[0, 0, 1, 0.96])
combined_top10_fig = OUT_DIR / "figures" / "top10_importance_grid_5models.png"
plt.savefig(combined_top10_fig, dpi=300)
plt.close()
print("✅ Saved:", combined_top10_fig)

# ============================================================
# PART 4) CANONICAL SPLIT METRICS + McNemar + DeLong + Calibration
# ============================================================
print_header("PART 4) CANONICAL SPLIT METRICS + TESTS + CALIBRATION")
print("Canonical seed:", canonical_seed)
print("X_train:", X_train.shape, "X_test:", X_test.shape)

models_final = build_models(seed=0)
preds = {}
scores = {}

for name in MODEL_ORDER:
    model = models_final[name]
    model.fit(X_train, y_train)
    preds[name] = model.predict(X_test)

    try:
        scores[name] = model.predict_proba(X_test)[:, 1]
    except Exception:
        try:
            s = model.decision_function(X_test)
            scores[name] = MinMaxScaler().fit_transform(s.reshape(-1, 1)).ravel()
        except Exception:
            scores[name] = None

rows = []
for name in MODEL_ORDER:
    row = {"Model": name}
    row["Accuracy"] = float(accuracy_score(y_test, preds[name]))
    row.update(compute_classification_metrics(y_test, preds[name]))

    if scores[name] is not None:
        row["ROC_AUC"] = float(roc_auc_score(y_test, scores[name]))
        row["PR_AUC"] = float(average_precision_score(y_test, scores[name]))
        row["Brier"] = float(brier_score_loss(y_test, scores[name]))
    else:
        row["ROC_AUC"] = np.nan
        row["PR_AUC"] = np.nan
        row["Brier"] = np.nan
    rows.append(row)

metrics_df = pd.DataFrame(rows).sort_values("Accuracy", ascending=False)
metrics_path = OUT_DIR / "tables" / "canonical_metrics_5models.csv"
metrics_df.to_csv(metrics_path, index=False)
print("✅ Saved:", metrics_path)

# McNemar
mcn_rows = []
for i in range(len(MODEL_ORDER)):
    for j in range(i+1, len(MODEL_ORDER)):
        A, B = MODEL_ORDER[i], MODEL_ORDER[j]
        stat, pval, table = mcnemar_test(y_test, preds[A], preds[B])
        mcn_rows.append({"A": A, "B": B, "statistic": stat, "p_value": pval, "table": str(table)})
mcn_df = pd.DataFrame(mcn_rows).sort_values("p_value")
mcn_path = OUT_DIR / "tables" / "mcnemar_canonical_5models.csv"
mcn_df.to_csv(mcn_path, index=False)
print("✅ Saved:", mcn_path)

# DeLong ROC-AUC tests
dl_rows = []
for i in range(len(MODEL_ORDER)):
    for j in range(i+1, len(MODEL_ORDER)):
        A, B = MODEL_ORDER[i], MODEL_ORDER[j]
        if scores[A] is None or scores[B] is None:
            dl_rows.append({"A": A, "B": B, "z_stat": np.nan, "p_value": np.nan, "AUC_A": np.nan, "AUC_B": np.nan})
            continue
        z, p, aucA, aucB = delong_roc_test(y_test.astype(int), scores[A], scores[B])
        dl_rows.append({"A": A, "B": B, "z_stat": z, "p_value": p, "AUC_A": aucA, "AUC_B": aucB})
dl_df = pd.DataFrame(dl_rows).sort_values("p_value")
dl_path = OUT_DIR / "tables" / "delong_canonical_5models.csv"
dl_df.to_csv(dl_path, index=False)
print("✅ Saved:", dl_path)

# Calibration curves
plt.figure(figsize=(8, 6))
for name in MODEL_ORDER:
    if scores[name] is None:
        continue
    prob_true, prob_pred = calibration_curve(y_test.astype(int), scores[name], n_bins=10)
    plt.plot(prob_pred, prob_true, marker="o", label=name)
plt.plot([0, 1], [0, 1], linestyle="--", label="Perfectly calibrated")
plt.title("Calibration Curves (Canonical Split) — 5 Models")
plt.xlabel("Mean predicted probability")
plt.ylabel("Fraction of positives")
plt.legend(fontsize=9)
plt.tight_layout()
cal_fig = OUT_DIR / "figures" / "calibration_curves_canonical_5models.png"
plt.savefig(cal_fig, dpi=300)
plt.close()
print("✅ Saved:", cal_fig)

# ============================================================
# PART 5) PCA SUMMARY (descriptive)
# ============================================================
print_header("PART 5) PCA SUMMARY (descriptive)")
from sklearn.decomposition import PCA
pca = PCA(n_components=0.90, random_state=0)
X_pca = pca.fit_transform(X)
pca_summary = {
    "original_shape": [int(X.shape[0]), int(X.shape[1])],
    "reduced_shape": [int(X_pca.shape[0]), int(X_pca.shape[1])],
    "n_components_kept": int(pca.n_components_),
    "explained_variance_sum": float(np.sum(pca.explained_variance_ratio_))
}
with open(OUT_DIR / "tables" / "pca_summary.json", "w") as f:
    json.dump(pca_summary, f, indent=2)
print("✅ Saved PCA summary:", OUT_DIR / "tables" / "pca_summary.json")

# ============================================================
# PART 6) REDUNDANCY METHODS (Top-10 features; standardized TRAIN only)
# ============================================================
print_header("PART 6) REDUNDANCY (Top-10 per model; standardized TRAIN only)")

def redundancy_variance(Z):
    per_sample_var = np.var(Z, axis=1, ddof=0)
    var_mean = float(np.mean(per_sample_var))
    red = 1.0 / (1.0 + var_mean)
    return float(red), float(var_mean)

def redundancy_mpad(Z):
    n, k = Z.shape
    pairs = []
    for a in range(k):
        for b in range(a+1, k):
            pairs.append(np.mean(np.abs(Z[:, a] - Z[:, b])))
    D = float(np.mean(pairs)) if len(pairs) else np.nan
    red = 1.0 / (1.0 + D)
    return float(red), float(D)

def redundancy_rmspd(Z):
    n, k = Z.shape
    pairs = []
    for a in range(k):
        for b in range(a+1, k):
            pairs.append(np.mean((Z[:, a] - Z[:, b])**2))
    ms = float(np.mean(pairs)) if len(pairs) else np.nan
    D = float(np.sqrt(ms)) if ms == ms else np.nan
    red = 1.0 / (1.0 + D)
    return float(red), float(D)

X_train_std = pd.DataFrame(
    StandardScaler().fit_transform(X_train),
    columns=X_train.columns,
    index=X_train.index
)

redundancy_rows = []
detail_rows = []

for name in MODEL_ORDER:
    top10 = importance_dfs[name]["Feature"].head(10).tolist()
    Z = X_train_std[top10].to_numpy()

    r_var, var_mean = redundancy_variance(Z)
    r_mpad, D_mpad = redundancy_mpad(Z)
    r_rmspd, D_rmspd = redundancy_rmspd(Z)

    redundancy_rows.append({
        "Model": name,
        "Red_Variance": r_var,
        "Red_MPAD": r_mpad,
        "Red_RMSPD": r_rmspd
    })
    detail_rows.append({
        "Model": name,
        "MeanWithinSampleVariance": var_mean,
        "MPAD_Distance": D_mpad,
        "RMSPD_Distance": D_rmspd
    })

redundancy_df = pd.DataFrame(redundancy_rows).merge(metrics_df, on="Model", how="left")
detail_df = pd.DataFrame(detail_rows)

red_path = OUT_DIR / "tables" / "redundancy_methods_5models.csv"
detail_path = OUT_DIR / "tables" / "redundancy_components_5models.csv"
redundancy_df.to_csv(red_path, index=False)
detail_df.to_csv(detail_path, index=False)
print("✅ Saved:", red_path)
print("✅ Saved:", detail_path)

# ============================================================
# PART 7) REDUNDANCY vs METRICS PLOTS (3 redundancy defs)
# ============================================================
print_header("PART 7) REDUNDANCY vs METRICS PLOTS")

metrics = ["Sensitivity", "Specificity", "F1", "Precision"]
redundancy_methods = [
    ("Red_Variance", "Variance-based redundancy"),
    ("Red_MPAD", "MPAD-based redundancy"),
    ("Red_RMSPD", "RMSPD-based redundancy"),
]

model_list_7 = redundancy_df["Model"].tolist()
palette = sns.color_palette("tab10", n_colors=len(model_list_7))
MODEL_COLORS = {model: palette[i] for i, model in enumerate(model_list_7)}

def linear_fit(x, y):
    x = np.asarray(x, float)
    y = np.asarray(y, float)
    b, a = np.polyfit(x, y, 1)
    yhat = a + b * x
    ss_res = np.sum((y - yhat)**2)
    ss_tot = np.sum((y - np.mean(y))**2)
    r2 = 1 - ss_res / ss_tot if ss_tot > 0 else np.nan
    return float(a), float(b), float(r2), yhat

for red_col, red_desc in redundancy_methods:
    fig, axes = plt.subplots(2, 2, figsize=(16, 12))
    axes = axes.ravel()

    legend_handles = [
        plt.Line2D([0], [0], marker='o', linestyle='',
                   markersize=10, color=MODEL_COLORS[m])
        for m in model_list_7
    ]

    for ax, metric in zip(axes, metrics):
        x = redundancy_df[red_col].values
        yvals = redundancy_df[metric].values

        for _, row in redundancy_df.iterrows():
            ax.scatter(row[red_col], row[metric], s=160, color=MODEL_COLORS[row["Model"]])

        a, b, r2, yhat = linear_fit(x, yvals)
        order = np.argsort(x)
        ax.plot(x[order], yhat[order], linewidth=2)

        ax.set_title(f"{metric} vs {red_col}\nLinear fit: R²={r2:.3f}", fontsize=14, weight="bold")
        ax.set_xlabel(red_col)
        ax.set_ylabel(metric)
        ax.grid(True)

    fig.legend(
        legend_handles,
        model_list_7,
        title="MODEL (dot color key)",
        loc="center right",
        bbox_to_anchor=(1.02, 0.5),
        frameon=True
    )
    plt.suptitle(f"{red_col} — {red_desc}\nRedundancy vs Classification Metrics (5 Models)", fontsize=16, weight="bold")
    plt.tight_layout(rect=[0, 0, 0.85, 0.93])

    outp = OUT_DIR / "figures" / f"redundancy_vs_metrics_{red_col}.png"
    plt.savefig(outp, dpi=300)
    plt.close()
    print("✅ Saved:", outp)

# ============================================================
# PART 8) FINAL MANIFEST (all key outputs)
# ============================================================
print_header("PART 8) SAVE FINAL MANIFEST")
final_manifest = {
    "data_path": str(DATA_PATH),
    "out_dir": str(OUT_DIR),
    "seeds": SEEDS,
    "split": "Stratified 80/20 train-test per seed",
    "canonical_seed": canonical_seed,
    "models": MODEL_ORDER,
    "tables": {
        "seed_accuracies": str(seed_acc_path),
        "accuracy_summary": str(acc_summary_path),
        "paired_ttests_accuracy": str(tt_path),
        "canonical_metrics": str(metrics_path),
        "mcnemar": str(mcn_path),
        "delong": str(dl_path),
        "top10_features": str(top10_path),
        "redundancy_methods": str(red_path),
        "redundancy_components": str(detail_path),
    },
    "figures_dir": str(OUT_DIR / "figures")
}
with open(OUT_DIR / "manifest_final.json", "w") as f:
    json.dump(final_manifest, f, indent=2)

print("✅ DONE — everything saved to:", OUT_DIR)
print("✅ Final manifest:", OUT_DIR / "manifest_final.json")


PART 0) LOAD CLEAN FROZEN DATA (AD03 Gaussian) + CREATE OUTPUT FOLDER
✅ Loaded: C:\Users\niuni\Research\Alzheimer\nacc_clean_complete_case_AD03.csv
Shape df: (35635, 82)
Shape X : (35635, 78)
Positive rate: 0.7217903746316824
Features: 78
✅ Output folder: C:\Users\niuni\Research\Alzheimer\5 model\run_AD03_gaussian_20260223_120938

PART 1) ACCURACY ACROSS 50 SEEDS (80/20 stratified) — SVM LAST


Part 1: 50 seeds (accuracy):   0%|          | 0/50 [00:00<?, ?it/s]

✅ Saved: C:\Users\niuni\Research\Alzheimer\5 model\run_AD03_gaussian_20260223_120938\tables\seed_accuracies_5models.csv
✅ Saved: C:\Users\niuni\Research\Alzheimer\5 model\run_AD03_gaussian_20260223_120938\tables\accuracy_summary_mean_sd_5models.csv

PART 2) ACCURACY BAR PLOT (5 models)


C:\Users\niuni\AppData\Local\Temp\ipykernel_17496\2478450240.py:314: FutureWarning: 

The `ci` parameter is deprecated. Use `errorbar=None` for the same effect.

  ax = sns.barplot(data=acc_summary, x="Model", y="mean_acc", palette="tab10", ci=None)
C:\Users\niuni\AppData\Local\Temp\ipykernel_17496\2478450240.py:314: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  ax = sns.barplot(data=acc_summary, x="Model", y="mean_acc", palette="tab10", ci=None)


✅ Saved: C:\Users\niuni\Research\Alzheimer\5 model\run_AD03_gaussian_20260223_120938\figures\accuracy_bar_5models.png
✅ Saved: C:\Users\niuni\Research\Alzheimer\5 model\run_AD03_gaussian_20260223_120938\tables\paired_ttests_accuracy_across_seeds_5models.csv

PART 3) FEATURE IMPORTANCE ACROSS 50 SEEDS (Top-10 per model)


Part 3: feature importance:   0%|          | 0/50 [00:00<?, ?it/s]

✅ Saved: C:\Users\niuni\Research\Alzheimer\5 model\run_AD03_gaussian_20260223_120938\tables\feature_importance_all_5models.csv
✅ Saved: C:\Users\niuni\Research\Alzheimer\5 model\run_AD03_gaussian_20260223_120938\tables\top10_features_5models.csv
✅ Saved: C:\Users\niuni\Research\Alzheimer\5 model\run_AD03_gaussian_20260223_120938\figures\top10_importance_XGBoost.png
✅ Saved: C:\Users\niuni\Research\Alzheimer\5 model\run_AD03_gaussian_20260223_120938\figures\top10_importance_Logistic_Regression.png
✅ Saved: C:\Users\niuni\Research\Alzheimer\5 model\run_AD03_gaussian_20260223_120938\figures\top10_importance_Ridge_Classifier.png
✅ Saved: C:\Users\niuni\Research\Alzheimer\5 model\run_AD03_gaussian_20260223_120938\figures\top10_importance_KNN.png
✅ Saved: C:\Users\niuni\Research\Alzheimer\5 model\run_AD03_gaussian_20260223_120938\figures\top10_importance_SVM_Linear.png
✅ Saved: C:\Users\niuni\Research\Alzheimer\5 model\run_AD03_gaussian_20260223_120938\figures\top10_importance_grid_5models.p

In [2]:
import numpy as np
import pandas as pd
from pathlib import Path

# Windows screenshot path:
# C:\Users\niuni\Research\Alzheimer\nacc_clean_complete_case_AD02.csv

BASE_DIR = Path.home() / "Research" / "Alzheimer"
IN_PATH  = BASE_DIR / "nacc_clean_complete_case_AD02.csv"
OUT_PATH = BASE_DIR / "nacc_clean_complete_case_AD02_dup_gauss.csv"

TARGET = "NACCALZD"

df = pd.read_csv(IN_PATH, low_memory=False)

# Keep definitive labels only (match your pipeline behavior)
df = df[df[TARGET].isin([0, 1])].copy()
df[TARGET] = df[TARGET].astype(int)

# Choose which columns to add noise to: numeric predictors only (exclude target)
num_cols = df.select_dtypes(include=[np.number]).columns.tolist()
num_cols = [c for c in num_cols if c != TARGET]

# --- Noise settings ---
# Option A (recommended): sigma is a fraction of each feature's SD (scale-aware)
sigma_frac = 0.10  # 10% of feature SD; try 0.01, 0.05, 0.10, 0.20
rng = np.random.default_rng(0)

sigmas = df[num_cols].std(ddof=0).replace(0, 1.0) * sigma_frac
noise = rng.normal(loc=0.0, scale=sigmas.values, size=(len(df), len(num_cols)))

dup = df[num_cols].to_numpy(dtype=float) + noise
dup_df = pd.DataFrame(
    dup,
    columns=[f"{c}__dup_gn{sigma_frac:g}" for c in num_cols],
    index=df.index
)

df_out = pd.concat([df, dup_df], axis=1)
df_out.to_csv(OUT_PATH, index=False)

print("✅ Read :", IN_PATH)
print("✅ Wrote:", OUT_PATH)
print("Original cols:", df.shape[1], "-> New cols:", df_out.shape[1])
print("Added dup cols:", dup_df.shape[1])

✅ Read : C:\Users\niuni\Research\Alzheimer\nacc_clean_complete_case_AD02.csv
✅ Wrote: C:\Users\niuni\Research\Alzheimer\nacc_clean_complete_case_AD02_dup_gauss.csv
Original cols: 43 -> New cols: 82
Added dup cols: 39


In [3]:
import sys
!{sys.executable} -m pip install numpy pandas scikit-learn matplotlib seaborn xgboost tqdm


  Using cached pandas-3.0.1-cp314-cp314-win_amd64.whl.metadata (19 kB)
  Using cached scikit_learn-1.8.0-cp314-cp314-win_amd64.whl.metadata (11 kB)
  Using cached matplotlib-3.10.8-cp314-cp314-win_amd64.whl.metadata (52 kB)
  Using cached seaborn-0.13.2-py3-none-any.whl.metadata (5.4 kB)
  Using cached xgboost-3.2.0-py3-none-win_amd64.whl.metadata (2.1 kB)
  Using cached scipy-1.17.1-cp314-cp314-win_amd64.whl.metadata (60 kB)
  Using cached joblib-1.5.3-py3-none-any.whl.metadata (5.5 kB)
  Using cached contourpy-1.3.3-cp314-cp314-win_amd64.whl.metadata (5.5 kB)
  Using cached cycler-0.12.1-py3-none-any.whl.metadata (3.8 kB)
  Using cached fonttools-4.61.1-cp314-cp314-win_amd64.whl.metadata (116 kB)
  Using cached kiwisolver-1.4.9-cp314-cp314-win_amd64.whl.metadata (6.4 kB)
Using cached pandas-3.0.1-cp314-cp314-win_amd64.whl (9.9 MB)
Using cached scikit_learn-1.8.0-cp314-cp314-win_amd64.whl (8.1 MB)
Using cached matplotlib-3.10.8-cp314-cp314-win_amd64.whl (8.3 MB)
Using cached seaborn-0


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [4]:
import numpy as np
import pandas as pd
from pathlib import Path

# Windows screenshot path:
# C:\Users\niuni\Research\Alzheimer\nacc_clean_complete_case_AD02.csv

BASE_DIR = Path.home() / "Research" / "Alzheimer"
IN_PATH  = BASE_DIR / "nacc_clean_complete_case_AD02.csv"
OUT_PATH = BASE_DIR / "nacc_clean_complete_case_AD02_dup_gauss.csv"

TARGET = "NACCALZD"

df = pd.read_csv(IN_PATH, low_memory=False)

# Keep definitive labels only (match your pipeline behavior)
df = df[df[TARGET].isin([0, 1])].copy()
df[TARGET] = df[TARGET].astype(int)

# Choose which columns to add noise to: numeric predictors only (exclude target)
num_cols = df.select_dtypes(include=[np.number]).columns.tolist()
num_cols = [c for c in num_cols if c != TARGET]

# --- Noise settings ---
# Option A (recommended): sigma is a fraction of each feature's SD (scale-aware)
sigma_frac = 0.10  # 10% of feature SD; try 0.01, 0.05, 0.10, 0.20
rng = np.random.default_rng(0)

sigmas = df[num_cols].std(ddof=0).replace(0, 1.0) * sigma_frac
noise = rng.normal(loc=0.0, scale=sigmas.values, size=(len(df), len(num_cols)))

dup = df[num_cols].to_numpy(dtype=float) + noise
dup_df = pd.DataFrame(
    dup,
    columns=[f"{c}__dup_gn{sigma_frac:g}" for c in num_cols],
    index=df.index
)

df_out = pd.concat([df, dup_df], axis=1)
df_out.to_csv(OUT_PATH, index=False)

print("✅ Read :", IN_PATH)
print("✅ Wrote:", OUT_PATH)
print("Original cols:", df.shape[1], "-> New cols:", df_out.shape[1])
print("Added dup cols:", dup_df.shape[1])

✅ Read : C:\Users\niuni\Research\Alzheimer\nacc_clean_complete_case_AD02.csv
✅ Wrote: C:\Users\niuni\Research\Alzheimer\nacc_clean_complete_case_AD02_dup_gauss.csv
Original cols: 43 -> New cols: 82
Added dup cols: 39


In [2]:
import numpy as np
import pandas as pd
from pathlib import Path

# Windows screenshot path:
# C:\Users\niuni\Research\Alzheimer\nacc_clean_complete_case_AD02.csv

BASE_DIR = Path.home() / "Research" / "Alzheimer"
IN_PATH  = BASE_DIR / "nacc_clean_complete_case_AD02.csv"
OUT_PATH = BASE_DIR / "nacc_clean_complete_case_AD02_dup_gauss.csv"

TARGET = "NACCALZD"

df = pd.read_csv(IN_PATH, low_memory=False)

# Keep definitive labels only (match your pipeline behavior)
df = df[df[TARGET].isin([0, 1])].copy()
df[TARGET] = df[TARGET].astype(int)

# Choose which columns to add noise to: numeric predictors only (exclude target)
num_cols = df.select_dtypes(include=[np.number]).columns.tolist()
num_cols = [c for c in num_cols if c != TARGET]

# --- Noise settings ---
# Option A (recommended): sigma is a fraction of each feature's SD (scale-aware)
sigma_frac = 0.10  # 10% of feature SD; try 0.01, 0.05, 0.10, 0.20
rng = np.random.default_rng(0)

sigmas = df[num_cols].std(ddof=0).replace(0, 1.0) * sigma_frac
noise = rng.normal(loc=0.0, scale=sigmas.values, size=(len(df), len(num_cols)))

dup = df[num_cols].to_numpy(dtype=float) + noise
dup_df = pd.DataFrame(
    dup,
    columns=[f"{c}__dup_gn{sigma_frac:g}" for c in num_cols],
    index=df.index
)

df_out = pd.concat([df, dup_df], axis=1)
df_out.to_csv(OUT_PATH, index=False)

print("✅ Read :", IN_PATH)
print("✅ Wrote:", OUT_PATH)
print("Original cols:", df.shape[1], "-> New cols:", df_out.shape[1])
print("Added dup cols:", dup_df.shape[1])

✅ Read : C:\Users\niuni\Research\Alzheimer\nacc_clean_complete_case_AD02.csv
✅ Wrote: C:\Users\niuni\Research\Alzheimer\nacc_clean_complete_case_AD02_dup_gauss.csv
Original cols: 43 -> New cols: 82
Added dup cols: 39


In [5]:
import sys
!{sys.executable} -m pip install numpy pandas scikit-learn matplotlib seaborn tqdm xgboost


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [6]:
python.exe -m pip install --upgrade pip

SyntaxError: invalid syntax (842801469.py, line 1)

In [1]:
import numpy as np
import pandas as pd

print("NumPy version:", np.__version__)
print("Pandas version:", pd.__version__)

NumPy version: 2.4.2
Pandas version: 3.0.1


In [ ]:
import sys
!{sys.executable} -m pip install --upgrade pip
!{sys.executable} -m pip install statsmodels

In [3]:
import sys
!{sys.executable} -m pip install numpy pandas scikit-learn matplotlib seaborn tqdm xgboost statsmodels ipywidgets

   ---------------------------------------- 0.0/9.6 MB ? eta -:--:--
   ------------------------- -------------- 6.0/9.6 MB 33.2 MB/s eta 0:00:01
   ---------------------------------------- 9.6/9.6 MB 35.4 MB/s  0:00:00
   ---------------------------------------- 0.0/914.9 kB ? eta -:--:--
   ---------------------------------------- 914.9/914.9 kB 41.0 MB/s  0:00:00
   ---------------------------------------- 0.0/2.2 MB ? eta -:--:--
   ---------------------------------------- 2.2/2.2 MB 45.7 MB/s  0:00:00

   -------- ------------------------------- 1/5 [patsy]
   -------- ------------------------------- 1/5 [patsy]
   ---------------- ----------------------- 2/5 [jupyterlab_widgets]
   ------------------------ --------------- 3/5 [statsmodels]
   ------------------------ --------------- 3/5 [statsmodels]
   ------------------------ --------------- 3/5 [statsmodels]
   ------------------------ --------------- 3/5 [statsmodels]
   ------------------------ --------------- 3/5 [statsmode

In [1]:
import statsmodels, numpy, pandas, sklearn
print("statsmodels:", statsmodels.__version__)
print("numpy:", numpy.__version__)
print("pandas:", pandas.__version__)
print("sklearn:", sklearn.__version__)

statsmodels: 0.14.6
numpy: 2.4.2
pandas: 3.0.1
sklearn: 1.8.0


In [3]:
# =========================
# CLEAN OOF HEATMAP RUNNER
# =========================
import numpy as np
import pandas as pd
from pathlib import Path

from sklearn.model_selection import StratifiedKFold
from sklearn.model_selection import cross_val_predict
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression, RidgeClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier

from xgboost import XGBClassifier

import matplotlib.pyplot as plt

# -------------------------
# PATHS (EDIT ONLY THESE)
# -------------------------
BASE_DIR = Path.home() / "Research" / "Alzheimer"              # <-- your folder
DATA_FILE = "nacc_clean_complete_case_AD03.csv"                # <-- your CSV
TARGET = "NACCALZD"

OUT_DIR = BASE_DIR / "heatmaps_release" / "oof_heatmaps_clean"
TABLE_DIR = OUT_DIR / "tables"
FIG_DIR = OUT_DIR / "figures"
for d in [OUT_DIR, TABLE_DIR, FIG_DIR]:
    d.mkdir(parents=True, exist_ok=True)

FINAL_X_PATH = BASE_DIR / DATA_FILE
print("Reading:", FINAL_X_PATH)
assert FINAL_X_PATH.exists(), f"File not found: {FINAL_X_PATH}"

# -------------------------
# LOAD DATA
# -------------------------
df = pd.read_csv(FINAL_X_PATH, low_memory=False)

# Fix common mixed-type issue (like your EDUC warning)
if "EDUC" in df.columns:
    df["EDUC"] = pd.to_numeric(df["EDUC"], errors="coerce")

# Keep numeric predictors only (like your earlier pipeline)
X_df = df.select_dtypes(include=[np.number]).copy()
assert TARGET in X_df.columns, f"TARGET '{TARGET}' not found in numeric cols. Check column name."
y = X_df[TARGET].astype(int).values
X_df = X_df.drop(columns=[TARGET])

print(f"Rows: {len(X_df):,} | Features: {X_df.shape[1]}")
print(f"Positive rate: {y.mean():.4f}")

# -------------------------
# CV SETUP
# -------------------------
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=0)

# -------------------------
# MODEL DEFINITIONS
# -------------------------
models = {
    "Logistic_Regression": Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("clf", LogisticRegression(max_iter=5000, n_jobs=None))
    ]),
    "Ridge_Classifier": Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("clf", RidgeClassifier())
    ]),
    "SVM_Linear": Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        # SVC gives probabilities only if probability=True; for speed we can use decision_function instead,
        # but for correlation heatmaps, either is fine. We'll use probability=True for simplicity.
        ("scaler", StandardScaler()),
        ("clf", SVC(kernel="linear", probability=True))
    ]),
    "KNN": Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),  # ✅ scaling happens here (no new dataset)
        ("clf", KNeighborsClassifier(n_neighbors=15))
    ]),
    "XGBoost": Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("clf", XGBClassifier(
            n_estimators=400,
            max_depth=4,
            learning_rate=0.05,
            subsample=0.8,
            colsample_bytree=0.8,
            reg_lambda=1.0,
            eval_metric="logloss",
            n_jobs=4,
            random_state=0
        ))
    ])
}

# -------------------------
# HELPER: OOF for single-feature models
# -------------------------
def get_single_feature_oof(model, X_onecol, y, cv):
    """
    Returns OOF predictions for a single feature.
    For classifiers that support predict_proba: use proba[:,1]
    Otherwise fall back to decision_function and rank-based outputs.
    """
    # cross_val_predict will refit model per fold
    if hasattr(model, "predict_proba"):
        p = cross_val_predict(model, X_onecol, y, cv=cv, method="predict_proba")[:, 1]
        return p
    # fallback: decision_function
    if hasattr(model, "decision_function"):
        s = cross_val_predict(model, X_onecol, y, cv=cv, method="decision_function")
        # convert to ranks 0..1 to make correlations stable
        r = pd.Series(s).rank(method="average", pct=True).values
        return r
    # final fallback: predict labels (not ideal, but avoids crashes)
    pred = cross_val_predict(model, X_onecol, y, cv=cv, method="predict")
    return pred.astype(float)

def save_corr_heatmap(corr, title, out_png, vlim=0.7):
    """
    Saves a simple heatmap. NaNs are shown as 0 for plotting so it never crashes.
    """
    corr_plot = corr.replace([np.inf, -np.inf], np.nan).fillna(0.0)

    fig = plt.figure(figsize=(10, 8))
    ax = plt.gca()
    im = ax.imshow(corr_plot.values, vmin=-vlim, vmax=vlim, aspect="auto")
    ax.set_title(title)
    ax.set_xticks([])
    ax.set_yticks([])
    plt.colorbar(im, fraction=0.046, pad=0.04)
    plt.tight_layout()
    fig.savefig(out_png, dpi=200)
    plt.close(fig)

# -------------------------
# MAIN LOOP
# -------------------------
feature_names = list(X_df.columns)

for model_name, model in models.items():
    print(f"\n=== {model_name} ===")

    # build OOF prediction matrix: rows = patients, cols = features
    oof_mat = np.zeros((len(X_df), len(feature_names)), dtype=float)

    for j, feat in enumerate(feature_names):
        X_one = X_df[[feat]].values  # single-column 2D array
        oof_mat[:, j] = get_single_feature_oof(model, X_one, y, cv)

        if (j + 1) % 10 == 0 or (j + 1) == len(feature_names):
            print(f"  done {j+1}/{len(feature_names)} features")

    oof_df = pd.DataFrame(oof_mat, columns=feature_names)

    # Save OOF predictions
    out_oof_csv = TABLE_DIR / f"oof_scores_{model_name}.csv"
    oof_df.to_csv(out_oof_csv, index=False)
    print("  saved:", out_oof_csv)

    # Correlation of prediction vectors (Spearman)
    corr_df = oof_df.corr(method="spearman")

    out_corr_csv = TABLE_DIR / f"heatmap_corr_{model_name}.csv"
    corr_df.to_csv(out_corr_csv)
    print("  saved:", out_corr_csv)

    # Save PNG (safe against NaN)
    out_png = FIG_DIR / f"heatmap_{model_name}_OOF_spearman.png"
    save_corr_heatmap(corr_df, f"{model_name}: OOF prediction correlation (Spearman)", out_png, vlim=0.7)
    print("  saved:", out_png)

print("\n✅ DONE. Outputs saved to:", OUT_DIR)
print(" - tables/: OOF scores + correlation matrices")
print(" - figures/: heatmap PNGs")

Reading: C:\Users\niuni\Research\Alzheimer\nacc_clean_complete_case_AD03.csv
Rows: 35,635 | Features: 79
Positive rate: 0.7218

=== Logistic_Regression ===
  done 10/79 features
  done 20/79 features
  done 30/79 features
  done 40/79 features
  done 50/79 features
  done 60/79 features
  done 70/79 features
  done 79/79 features
  saved: C:\Users\niuni\Research\Alzheimer\heatmaps_release\oof_heatmaps_clean\tables\oof_scores_Logistic_Regression.csv
  saved: C:\Users\niuni\Research\Alzheimer\heatmaps_release\oof_heatmaps_clean\tables\heatmap_corr_Logistic_Regression.csv
  saved: C:\Users\niuni\Research\Alzheimer\heatmaps_release\oof_heatmaps_clean\figures\heatmap_Logistic_Regression_OOF_spearman.png

=== Ridge_Classifier ===
  done 10/79 features
  done 20/79 features
  done 30/79 features
  done 40/79 features
  done 50/79 features
  done 60/79 features
  done 70/79 features
  done 79/79 features
  saved: C:\Users\niuni\Research\Alzheimer\heatmaps_release\oof_heatmaps_clean\tables\oof_

KeyboardInterrupt: 